In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("income.csv")
df.head()

,age,sex,education,education-num,marital-status,workclass,occupation,hours-per-week,income,label
0,27,Male,HS-grad,9,Never-married,Private,Craft-repair,40,<=50K,0
1,47,Male,Masters,14,Married,Local-gov,Exec-managerial,50,>50K,1
2,59,Male,HS-grad,9,Divorced,Self-emp,Prof-specialty,20,<=50K,0
3,38,Female,Prof-school,15,Never-married,Federal-gov,Prof-specialty,57,>50K,1
4,64,Female,11th,7,Widowed,Private,Farming-fishing,40,<=50K,0


In [10]:
categorical_columns = [
    'sex',
    'education',
    'marital-status',
    'workclass',
    'occupation'
]
continuous_columns = [
    'age',
    'education-num',
    'hours-per-week'
]
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [11]:
df[label_column] = df[label_column].map({'<=50K':0, '>50K':1})

In [12]:
categorical_data = np.stack([df[col].values for col in categorical_columns], axis=1)
continuous_data = np.stack([df[col].values for col in continuous_columns], axis=1)

labels = df[label_column].values

In [13]:
categorical_data = torch.tensor(categorical_data, dtype=torch.long)
continuous_data = torch.tensor(continuous_data, dtype=torch.float)
labels = torch.tensor(labels, dtype=torch.long)

In [14]:
train_categorical = categorical_data[:25000]
test_categorical = categorical_data[25000:]

train_continuous = continuous_data[:25000]
test_continuous = continuous_data[25000:]

train_labels = labels[:25000]
test_labels = labels[25000:]

In [15]:
batch_size = 64

train_dataset = TensorDataset(train_categorical, train_continuous, train_labels)
test_dataset = TensorDataset(test_categorical, test_continuous, test_labels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [16]:
class TabularModel(nn.Module):

    def __init__(self, embedding_sizes, n_continuous, hidden_size=50, p=0.4):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(categories, size) 
            for categories, size in embedding_sizes
        ])

        self.embedding_dropout = nn.Dropout(p)

        self.batch_norm_cont = nn.BatchNorm1d(n_continuous)

        n_emb = sum(e.embedding_dim for e in self.embeddings)

        self.layer1 = nn.Linear(n_emb + n_continuous, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p)

        self.output = nn.Linear(hidden_size, 2)

    def forward(self, x_cat, x_cont):

        embeddings = [emb(x_cat[:,i]) for i,emb in enumerate(self.embeddings)]
        x = torch.cat(embeddings, 1)

        x = self.embedding_dropout(x)

        x_cont = self.batch_norm_cont(x_cont)

        x = torch.cat([x, x_cont], 1)

        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.output(x)

        return x

In [17]:
embedding_sizes = []

for col in categorical_columns:
    n_categories = df[col].nunique()
    emb_size = min(50, (n_categories+1)//2)
    embedding_sizes.append((n_categories, emb_size))

In [18]:
torch.manual_seed(42)

model = TabularModel(
    embedding_sizes,
    n_continuous=len(continuous_columns),
    hidden_size=50,
    p=0.4
)

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 300

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for cat, cont, y in train_loader:

        optimizer.zero_grad()

        y_pred = model(cat, cont)

        loss = criterion(y_pred, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if epoch % 25 == 0:
        print(f"Epoch {epoch} Loss {total_loss:.4f}")

Epoch 0 Loss 154.5793
Epoch 25 Loss 107.0155
Epoch 50 Loss 106.3901
Epoch 75 Loss 105.7193
Epoch 100 Loss 104.8878
Epoch 125 Loss 105.4888
Epoch 150 Loss 103.1153
Epoch 175 Loss 103.7408
Epoch 200 Loss 104.6526
Epoch 225 Loss 104.1206


In [ ]:
model.eval()

correct = 0
total = 0
test_loss = 0

with torch.no_grad():

    for cat, cont, y in test_loader:

        outputs = model(cat, cont)

        loss = criterion(outputs, y)
        test_loss += loss.item()

        _, predicted = torch.max(outputs,1)

        correct += (predicted == y).sum().item()
        total += y.size(0)

accuracy = correct/total

print("Test Loss:", test_loss)
print("Test Accuracy:", accuracy)